### Draw main graph

In [1]:
import numpy as np
import tensorflow.compat.v1 as tf
from config import *
from GPT_Model import *
from data_pipeline import *

tf.reset_default_graph()

tf.compat.v1.disable_eager_execution()
X = tf.placeholder(tf.int32, [None, hparams.n_time])
Y = tf.placeholder(tf.int32, [None, hparams.n_time])


logits = model(hparams, X)['logits']
cross_entropy = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=Y, logits=logits)
loss = tf.reduce_mean(cross_entropy)

global_step = tf.Variable(0, name='global_step')
learning_rate = tf.Variable(1e-4, name='learning_rate')

if mode == "pretrain":
    train_step = tf.train.AdamOptimizer(learning_rate).minimize(loss, global_step)
elif mode == "finetune":
    optimizer = tf.train.AdamOptimizer(learning_rate)
    output_vars = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope='linear1|linear2')
    train_step = optimizer.minimize(loss, var_list=output_vars, global_step=global_step)

# GPU number to use
gpu_options = tf.GPUOptions(visible_device_list="0")
sess = tf.Session(config=tf.ConfigProto(gpu_options=gpu_options))

sess.run(tf.global_variables_initializer())

print('graph create')
print(type)
print(mode)

Instructions for updating:
If using Keras pass *_constraint arguments to layers.
graph create
music
finetune


### Load model if exist && TensorboardX Logger

In [2]:
import tf_slim as slim
from tensorflow.python import pywrap_tensorflow

load_dir = '../pattern_save_model'
save_dir ='../pattern_final_save_model'

# only restore paremeters of transformer
sess.run(tf.global_variables_initializer())
ref_vars = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope='transformer')
saver = tf.train.Saver(ref_vars)

restore_file = tf.train.latest_checkpoint(load_dir)
print(restore_file)
if restore_file is not None:
    saver.restore(sess, restore_file)
    print("Model restored.", restore_file)
else:
    print('model not exist.')

# Logger
from tensorboardX import SummaryWriter

class Logger(SummaryWriter):
    def __init__(self, logdir):
        super(Logger, self).__init__(logdir)

    def log(self, log_string, value, iteration):
            self.add_scalar(log_string, value, iteration)
            
logger = Logger(save_dir)  
        

../pattern_save_model/checkpoint-2500
INFO:tensorflow:Restoring parameters from ../pattern_save_model/checkpoint-2500
Model restored. ../pattern_save_model/checkpoint-2500


### Train

In [3]:
import math

print('iteration\t', 'loss\t', 'train_perplexity\t')
while(True):
    for _ in range(100):
        _inputs = []
        _targets = []
        for _ in range(batch_size):
            while(True):
                x, y = get_data(hparams.n_time, data_train_files,'train', 0, type, EventDim)
                if(x.shape == y.shape):
                    break
                 
            _inputs.append(x)
            _targets.append(y)
        _inputs = np.stack(_inputs)
        _targets = np.stack(_targets)
        
        _, _global_step, _loss = sess.run([train_step, global_step, loss], 
                                          feed_dict={X: _inputs, 
                                                     Y: _targets,
                                                     learning_rate: 1e-4})
        
        train_perplexity = math.exp(_loss) # log perplexity is equal to perplexity 
        
        if _global_step % 10 == 0:
            logger.log('loss', _loss, _global_step)
            print(str(_global_step)+'\t', str(_loss)+'\t', str(train_perplexity)+'\t')
        
        if _global_step % 100 == 0:
            save_path = saver.save(sess, save_dir + '/checkpoint', global_step=_global_step)
            print("Model saved in path: %s" % save_path)

iteration	 loss	 train_perplexity	
0	 5.8852057	 359.6767623305039	
Model saved in path: ../pattern_final_save_model/checkpoint-0
10	 4.9802175	 145.5060194894975	
20	 4.580454	 97.55866337207844	
30	 4.377957	 79.67508022596253	
40	 4.428103	 83.77234743227555	
50	 4.3211784	 75.27728552758443	
60	 4.1788855	 65.29304093306654	
70	 4.2533073	 70.33765880606958	
80	 4.1035457	 60.55461382794475	
90	 3.9305935	 50.937199449236445	
100	 3.7614546	 43.010943452706215	
Model saved in path: ../pattern_final_save_model/checkpoint-100
110	 3.6759582	 39.48647296154659	
120	 3.6799073	 39.64271989154541	
130	 3.61211	 37.04412981028831	
140	 3.5819378	 35.94312362943626	
150	 3.512353	 33.52706232875135	
160	 3.6285098	 37.656657321496695	
170	 3.5164857	 33.66590792609597	
180	 3.508933	 33.412599765087265	
190	 3.521485	 33.83463866346778	
200	 3.4604712	 31.83197071885964	
Model saved in path: ../pattern_final_save_model/checkpoint-200
210	 3.469938	 32.13475130679667	
220	 3.4521363	 31.56

KeyboardInterrupt: 

### Compute perplexity on test set

In [4]:
import math

inputs = []
targets = []

l = len(data_test_files)

for i in range(l): 
    while(True):
        x_test, y_test = get_data(hparams.n_time, data_test_files, 'test', i, 'music', EventDim)
        if(x_test.shape == y_test.shape):
            break       
    inputs.append(x_test)
    targets.append(y_test)
    
inputs = np.stack(inputs)
targets = np.stack(targets)

test_loss = sess.run(loss, feed_dict={X: inputs, Y: targets})
test_perplexity = math.exp(test_loss)
test_perplexity

14.838651225519365